In [1]:
# libraries for this lab
import csv
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SID4 = 670
SEED = SID4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)


device: cuda


# Sneha Singh
## DATA 266 — Lab 1, Part 3
CycleGAN photo ↔ Monet (+ Kaggle submit)


## Part 3. CycleGAN

I train an unpaired CycleGAN: photo domain ↔ Monet domain.

- Two generators (`G_AB` photo→Monet, `G_BA` Monet→photo) and two PatchGAN discriminators
- LSGAN (MSE) adversarial loss + cycle-consistency (L1) + identity loss
- Image buffer for D updates; config-driven smoke vs full

Integrity: Kaggle images come only from my own model's inference — no pretrained image models on outputs.


## 3.0 Paths and config

Numbers live in `config.json`. Prints use repo-relative paths only.


In [2]:
# folders: notebook may be opened from src/ or Lab-1 root
cwd = Path.cwd()
if (cwd / "config.json").exists():
    SRC = cwd
elif (cwd / "task3_gan" / "sneha_singh" / "src" / "config.json").exists():
    SRC = cwd / "task3_gan" / "sneha_singh" / "src"
else:
    SRC = cwd

MEMBER = SRC.parent
TASK = MEMBER.parent
REPO = TASK.parent
DATA_RAW = TASK / "data"
MONET_DIR = DATA_RAW / "monet_jpg"
PHOTO_DIR = DATA_RAW / "photo_jpg"
CKPT = MEMBER / "checkpoints"
OUT = MEMBER / "outputs"
PRED_A2B = OUT / "pred_A2B"
PRED_B2A = OUT / "pred_B2A"
LOG_DIR = REPO / "reproducibility" / "raw_logs" / "sneha_singh" / "task3_gan"

for d in [DATA_RAW, MONET_DIR, PHOTO_DIR, CKPT, OUT, PRED_A2B, PRED_B2A,
          OUT / "samples", OUT / "loss_curves", OUT / "human_audit", LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with open(SRC / "config.json") as f:
    CFG = json.load(f)

SMOKE = bool(CFG["smoke"])
IMG = int(CFG["image_size"])
LOAD = int(CFG["load_size"])
BATCH = int(CFG["batch_size"])
LR = float(CFG["learning_rate"])
BETA1 = float(CFG["beta1"])
L_CYCLE = float(CFG["lambda_cycle"])
L_ID = float(CFG["lambda_identity"])
POOL = int(CFG["pool_size"])
SMOOTH = float(CFG["label_smoothing"])
N_RES = int(CFG["n_res_blocks"])
NGF = int(CFG["ngf"])
NDF = int(CFG["ndf"])

if SMOKE:
    N_EPOCHS = int(CFG["epochs_smoke"])
    N_MONET = int(CFG["n_monet_smoke"])
    N_PHOTO = int(CFG["n_photo_smoke"])
    N_SUBMIT = int(CFG["n_submit_smoke"])
    PHOTOS_PER_EPOCH = int(CFG.get("photos_per_epoch_smoke", N_PHOTO))
else:
    N_EPOCHS = int(CFG["n_epochs_const"]) + int(CFG["n_epochs_decay"])
    N_MONET = int(CFG["n_monet_full"])
    N_PHOTO = int(CFG["n_photo_full"])
    N_SUBMIT = int(CFG["n_submit_full"])
    PHOTOS_PER_EPOCH = int(CFG.get("photos_per_epoch_full", 400))


def relpath(p):
    p = Path(p).resolve()
    try:
        return str(p.relative_to(REPO.resolve()))
    except ValueError:
        return p.name


print("SRC:", relpath(SRC))
print("MONET_DIR:", relpath(MONET_DIR), "PHOTO_DIR:", relpath(PHOTO_DIR))
print("SMOKE:", SMOKE, "epochs:", N_EPOCHS, "batch:", BATCH, "img:", IMG)
print("photos_per_epoch:", PHOTOS_PER_EPOCH, "(reshuffled each epoch; full photos still used at submit)")
print("lambda_cycle:", L_CYCLE, "lambda_identity:", L_ID, "pool:", POOL)


SRC: task3_gan/sneha_singh/src
MONET_DIR: task3_gan/data/monet_jpg PHOTO_DIR: task3_gan/data/photo_jpg
SMOKE: False epochs: 80 batch: 4 img: 256
photos_per_epoch: 400 (reshuffled each epoch; full photos still used at submit)
lambda_cycle: 10.0 lambda_identity: 0.5 pool: 50


### 3.0.1 Dataset

I need unpaired JPEGs in:

- `task3_gan/data/monet_jpg/` (~300 Monet paintings, 256×256)
- `task3_gan/data/photo_jpg/` (~7038 photos, 256×256)

**Download once from Kaggle** (class competition dataset, or `gan-getting-started`):

```bash
# after: pip install kaggle  + place ~/.kaggle/kaggle.json
kaggle competitions download -c gan-getting-started -p /tmp/gan
unzip /tmp/gan/gan-getting-started.zip -d /tmp/gan
# copy jpg folders into task3_gan/data/
```

Raw images stay local / Drive — not pushed to GitHub.


In [3]:
# list images; stop with a clear message if data is missing
def list_images(folder):
    exts = {".jpg", ".jpeg", ".png"}
    return sorted([p for p in Path(folder).iterdir() if p.suffix.lower() in exts])


monet_paths = list_images(MONET_DIR)
photo_paths = list_images(PHOTO_DIR)
print("monet images:", len(monet_paths))
print("photo images:", len(photo_paths))

if len(monet_paths) == 0 or len(photo_paths) == 0:
    raise FileNotFoundError(
        "Missing Monet/Photo JPGs under task3_gan/data/. "
        "Download from Kaggle (gan-getting-started or class competition) "
        "into monet_jpg/ and photo_jpg/, then re-run."
    )

if N_MONET > 0:
    monet_paths = monet_paths[:N_MONET]
if N_PHOTO > 0:
    photo_paths = photo_paths[:N_PHOTO]
print("using monet:", len(monet_paths), "photo:", len(photo_paths))


monet images: 300
photo images: 7038
using monet: 300 photo: 7038


## 3.1 Model and training

Paper recipe (Zhu et al. CycleGAN):

- Generator: Johnson ResNet — **9** residual blocks at 256×256, reflection pad, instance norm, tanh
- Discriminator: PatchGAN **C64-C128-C256-C512** (no InstanceNorm on first C64), 70×70 field
- Losses: LSGAN (MSE) + cycle L1 (λ=10) + identity (0.5λ); **D objective ÷ 2**
- Image pool size 50; Adam lr=2e-4, β1=0.5

**Compute budget (lab session):**
- `n_steps = min(len(monet), len(photo))` each epoch (TensorFlow tutorial style — pair to shorter domain)
- batch 4, **40+40 epochs**, plus ~400 photos/epoch reshuffle as extra safety
- Full photo set still used when building Kaggle `images.zip`


In [4]:
# dataset + transforms (286 resize → random crop 256 + flip when training)
class ImageFolderDS(Dataset):
    def __init__(self, paths, train=True):
        self.paths = list(paths)
        if train:
            self.tf = transforms.Compose([
                transforms.Resize(LOAD, interpolation=transforms.InterpolationMode.BICUBIC),
                transforms.RandomCrop(IMG),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((IMG, IMG), interpolation=transforms.InterpolationMode.BICUBIC),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img)


class ImagePool:
    """History buffer of previously generated images (CycleGAN paper)."""

    def __init__(self, pool_size):
        self.pool_size = pool_size
        self.images = []

    def query(self, images):
        if self.pool_size == 0:
            return images
        out = []
        for img in images:
            img = torch.unsqueeze(img.data, 0)
            if len(self.images) < self.pool_size:
                self.images.append(img)
                out.append(img)
            else:
                if random.random() > 0.5:
                    idx = random.randint(0, self.pool_size - 1)
                    tmp = self.images[idx].clone()
                    self.images[idx] = img
                    out.append(tmp)
                else:
                    out.append(img)
        return torch.cat(out, dim=0)


def conv_norm_relu(in_ch, out_ch, k=3, s=1, p=1, norm=True, relu=True, transpose=False):
    layers = []
    if transpose:
        layers.append(nn.ConvTranspose2d(in_ch, out_ch, k, s, p, output_padding=1, bias=False))
    else:
        layers.append(nn.Conv2d(in_ch, out_ch, k, s, p, bias=not norm))
    if norm:
        layers.append(nn.InstanceNorm2d(out_ch, affine=False, track_running_stats=False))
    if relu:
        layers.append(nn.ReLU(inplace=True))
    return layers


class ResnetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3, bias=False),
            nn.InstanceNorm2d(dim, affine=False, track_running_stats=False),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3, bias=False),
            nn.InstanceNorm2d(dim, affine=False, track_running_stats=False),
        )

    def forward(self, x):
        return x + self.block(x)


class ResnetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_blocks=9):
        super().__init__()
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, ngf, 7, bias=False),
            nn.InstanceNorm2d(ngf, affine=False, track_running_stats=False),
            nn.ReLU(True),
        ]
        # down
        n = ngf
        for _ in range(2):
            model += [
                nn.Conv2d(n, n * 2, 3, 2, 1, bias=False),
                nn.InstanceNorm2d(n * 2, affine=False, track_running_stats=False),
                nn.ReLU(True),
            ]
            n *= 2
        for _ in range(n_blocks):
            model += [ResnetBlock(n)]
        # up
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(n, n // 2, 3, 2, 1, output_padding=1, bias=False),
                nn.InstanceNorm2d(n // 2, affine=False, track_running_stats=False),
                nn.ReLU(True),
            ]
            n //= 2
        model += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, out_ch, 7), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)



class NLayerDiscriminator(nn.Module):
    """PatchGAN C64-C128-C256-C512 (70x70). No InstanceNorm on first C64 (paper)."""

    def __init__(self, in_ch=3, ndf=64, n_layers=3):
        super().__init__()
        kw, pad = 4, 1
        # C64 — no InstanceNorm
        seq = [nn.Conv2d(in_ch, ndf, kw, 2, pad), nn.LeakyReLU(0.2, True)]
        nf_mult = 1
        for n in range(1, n_layers):
            nf_mult_prev, nf_mult = nf_mult, min(2 ** n, 8)
            seq += [
                nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kw, 2, pad, bias=False),
                nn.InstanceNorm2d(ndf * nf_mult, affine=False, track_running_stats=False),
                nn.LeakyReLU(0.2, True),
            ]
        # last C512 block stride 1
        nf_mult_prev, nf_mult = nf_mult, min(2 ** n_layers, 8)
        seq += [
            nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kw, 1, pad, bias=False),
            nn.InstanceNorm2d(ndf * nf_mult, affine=False, track_running_stats=False),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * nf_mult, 1, kw, 1, pad),
        ]
        self.model = nn.Sequential(*seq)

    def forward(self, x):
        return self.model(x)


print("model classes ready")


model classes ready


In [5]:
# build models, loaders, optimizers, pools
monet_ds = ImageFolderDS(monet_paths, train=True)
monet_loader = DataLoader(
    monet_ds, batch_size=BATCH, shuffle=True,
    num_workers=int(CFG["num_workers"]), drop_last=True,
)

def make_photo_loader(epoch_seed):
    """Resample ~PHOTOS_PER_EPOCH photos each epoch (budget fix; still covers full set over training)."""
    rng = random.Random(SEED + int(epoch_seed))
    paths = list(photo_paths)
    if PHOTOS_PER_EPOCH > 0 and PHOTOS_PER_EPOCH < len(paths):
        paths = rng.sample(paths, PHOTOS_PER_EPOCH)
    ds = ImageFolderDS(paths, train=True)
    return DataLoader(
        ds, batch_size=BATCH, shuffle=True,
        num_workers=int(CFG["num_workers"]), drop_last=True,
    )

photo_loader = make_photo_loader(0)

G_AB = ResnetGenerator(ngf=NGF, n_blocks=N_RES).to(device)  # photo -> monet
G_BA = ResnetGenerator(ngf=NGF, n_blocks=N_RES).to(device)  # monet -> photo
D_A = NLayerDiscriminator(ndf=NDF).to(device)               # monet disc
D_B = NLayerDiscriminator(ndf=NDF).to(device)               # photo disc

opt_G = torch.optim.Adam(
    list(G_AB.parameters()) + list(G_BA.parameters()), lr=LR, betas=(BETA1, 0.999)
)
opt_D = torch.optim.Adam(
    list(D_A.parameters()) + list(D_B.parameters()), lr=LR, betas=(BETA1, 0.999)
)

# linear LR decay over second half (full); smoke keeps constant LR
n_const = int(CFG["epochs_smoke"]) if SMOKE else int(CFG["n_epochs_const"])
n_decay = 0 if SMOKE else int(CFG["n_epochs_decay"])


def lr_lambda(epoch):
    # epoch is 0-based step count for LambdaLR
    if epoch < n_const:
        return 1.0
    return max(0.0, 1.0 - (epoch - n_const) / max(1, n_decay))


sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda=lr_lambda)
sched_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_lambda=lr_lambda)

pool_A = ImagePool(POOL)
pool_B = ImagePool(POOL)

criterion_gan = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_id = nn.L1Loss()

param_count = sum(p.numel() for m in [G_AB, G_BA, D_A, D_B] for p in m.parameters())
print("param_count:", param_count)
print("steps/epoch ~", min(len(monet_loader), len(photo_loader)), "(min of domains)", "| monet_batches", len(monet_loader), "| photo_batches", len(photo_loader))


param_count: 28273544
steps/epoch ~ 75 (min of domains) | monet_batches 75 | photo_batches 100


In [6]:
# train CycleGAN; log raw losses; keep best by cycle loss
log_path = LOG_DIR / ("train_smoke.log" if SMOKE else "train_full.log")
ckpt_path = CKPT / ("best_smoke.pt" if SMOKE else "best.pt")

history = {"g": [], "d": [], "cycle": [], "identity": [], "grad": [], "nan": []}
best_cycle = float("inf")
nan_count = 0
total_images = 0
t0 = time.time()
peak_mem = 0.0

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

# AMP on CUDA only (same idea as Part 1) — faster full runs on GPU lab / Colab
use_amp = device.type == "cuda"
amp_dtype = torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
print("AMP:", use_amp)

with open(log_path, "w") as logf:
    logf.write("member=sneha_singh smoke=%s device=%s\n" % (SMOKE, device))
    logf.write("epochs=%s batch=%s lr=%s lambda_cycle=%s lambda_id=%s pool=%s\n" % (
        N_EPOCHS, BATCH, LR, L_CYCLE, L_ID * L_CYCLE, POOL))

    for epoch in range(1, N_EPOCHS + 1):
        G_AB.train(); G_BA.train(); D_A.train(); D_B.train()
        sum_g = sum_d = sum_c = sum_i = sum_grad = 0.0
        steps = 0
        photo_loader = make_photo_loader(epoch)  # new photo subset each epoch
        monet_iter = iter(monet_loader)
        photo_iter = iter(photo_loader)
        n_steps = min(len(monet_loader), len(photo_loader))  # TF tutorial style: pair to shorter domain

        for _ in range(n_steps):
            try:
                real_A = next(monet_iter).to(device)   # monet
            except StopIteration:
                monet_iter = iter(monet_loader)
                real_A = next(monet_iter).to(device)
            try:
                real_B = next(photo_iter).to(device)   # photo
            except StopIteration:
                photo_iter = iter(photo_loader)
                real_B = next(photo_iter).to(device)

            # ---- Generators ----
            opt_G.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                fake_B = G_BA(real_A)          # monet -> photo
                fake_A = G_AB(real_B)          # photo -> monet
                rec_A = G_AB(fake_B)
                rec_B = G_BA(fake_A)
                id_A = G_AB(real_A)
                id_B = G_BA(real_B)
                loss_id = (criterion_id(id_A, real_A) + criterion_id(id_B, real_B)) * (L_ID * L_CYCLE)
                pred_fake_A = D_A(fake_A)
                pred_fake_B = D_B(fake_B)
                loss_gan = criterion_gan(pred_fake_A, torch.ones_like(pred_fake_A)) + \
                           criterion_gan(pred_fake_B, torch.ones_like(pred_fake_B))
                loss_cycle = (criterion_cycle(rec_A, real_A) + criterion_cycle(rec_B, real_B)) * L_CYCLE
                loss_G = loss_gan + loss_cycle + loss_id
            scaler.scale(loss_G).backward()
            scaler.unscale_(opt_G)
            grad_norm = 0.0
            for p in list(G_AB.parameters()) + list(G_BA.parameters()):
                if p.grad is not None:
                    grad_norm += float(p.grad.data.norm(2).item())
            scaler.step(opt_G)

            # ---- Discriminators ----
            opt_D.zero_grad(set_to_none=True)
            fake_A_d = pool_A.query(fake_A.detach())
            fake_B_d = pool_B.query(fake_B.detach())
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                pred_real_A = D_A(real_A)
                pred_real_B = D_B(real_B)
                pred_fake_A_d = D_A(fake_A_d)
                pred_fake_B_d = D_B(fake_B_d)
                real_label_A = torch.ones_like(pred_real_A) * SMOOTH
                real_label_B = torch.ones_like(pred_real_B) * SMOOTH
                # paper: divide each D objective by 2 to slow D vs G
                loss_D_A = 0.5 * (
                    criterion_gan(pred_real_A, real_label_A) +
                    criterion_gan(pred_fake_A_d, torch.zeros_like(pred_fake_A_d))
                )
                loss_D_B = 0.5 * (
                    criterion_gan(pred_real_B, real_label_B) +
                    criterion_gan(pred_fake_B_d, torch.zeros_like(pred_fake_B_d))
                )
                loss_D = loss_D_A + loss_D_B
            scaler.scale(loss_D).backward()
            scaler.step(opt_D)
            scaler.update()  # once per iter after both G and D steps

            if not torch.isfinite(loss_G) or not torch.isfinite(loss_D):
                nan_count += 1

            sum_g += float(loss_G.detach())
            sum_d += float(loss_D.detach())
            sum_c += float(loss_cycle.detach())
            sum_i += float(loss_id.detach())
            sum_grad += grad_norm
            steps += 1
            total_images += real_A.size(0) + real_B.size(0)

        sched_G.step(); sched_D.step()
        avg_g, avg_d = sum_g / steps, sum_d / steps
        avg_c, avg_i = sum_c / steps, sum_i / steps
        avg_grad = sum_grad / steps
        history["g"].append(avg_g); history["d"].append(avg_d)
        history["cycle"].append(avg_c); history["identity"].append(avg_i)
        history["grad"].append(avg_grad); history["nan"].append(nan_count)

        line = "epoch %d g=%.4f d=%.4f cycle=%.4f id=%.4f grad=%.4f nan=%d\n" % (
            epoch, avg_g, avg_d, avg_c, avg_i, avg_grad, nan_count)
        print(line.strip()); logf.write(line); logf.flush()

        if avg_c < best_cycle:
            best_cycle = avg_c
            torch.save({
                "G_AB": G_AB.state_dict(),
                "G_BA": G_BA.state_dict(),
                "D_A": D_A.state_dict(),
                "D_B": D_B.state_dict(),
                "epoch": epoch,
                "best_cycle": best_cycle,
                "smoke": SMOKE,
                "param_count": param_count,
            }, ckpt_path)

        if device.type == "cuda":
            peak_mem = max(peak_mem, torch.cuda.max_memory_allocated() / (1024 ** 2))

train_time = time.time() - t0
print("train_time_sec:", round(train_time, 1))
print("saved best:", relpath(ckpt_path), "best_cycle:", best_cycle)
print("raw log:", relpath(log_path))


AMP: True


/tmp/ipykernel_4969/512737859.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


epoch 1 g=10.2355 d=0.6764 cycle=6.4188 id=3.0324 grad=inf nan=0


epoch 2 g=9.0147 d=0.4862 cycle=5.6816 id=2.6257 grad=116.6299 nan=0


epoch 3 g=8.4474 d=0.5196 cycle=5.2361 id=2.3863 grad=inf nan=0


...

epoch 77 g=4.4266 d=0.2586 cycle=2.3069 id=1.1532 grad=116.4633 nan=0


epoch 78 g=4.4292 d=0.2499 cycle=2.2849 id=1.1435 grad=112.2116 nan=0


epoch 79 g=4.3704 d=0.2636 cycle=2.2411 id=1.1287 grad=111.6853 nan=0


epoch 80 g=4.3339 d=0.2601 cycle=2.2076 id=1.1137 grad=107.6565 nan=0


train_time_sec: 5238.1
saved best: task3_gan/sneha_singh/checkpoints/best.pt best_cycle: 2.207583204905192
raw log: reproducibility/raw_logs/sneha_singh/task3_gan/train_full.log


In [7]:
# loss curves
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
epochs = list(range(1, len(history["g"]) + 1))
ax[0].plot(epochs, history["g"], label="G")
ax[0].plot(epochs, history["d"], label="D")
ax[0].plot(epochs, history["cycle"], label="cycle")
ax[0].plot(epochs, history["identity"], label="identity")
ax[0].set_title("CycleGAN losses"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].plot(epochs, history["grad"], label="G grad norm")
ax[1].set_title("Gradient norm"); ax[1].legend(); ax[1].set_xlabel("epoch")
fig.tight_layout()
curve = OUT / "loss_curves" / ("losses_smoke.png" if SMOKE else "losses.png")
fig.savefig(curve, dpi=120)
plt.close(fig)
print("saved", relpath(curve))


saved task3_gan/sneha_singh/outputs/loss_curves/losses.png


## 3.2 Inference, samples, Kaggle zip

Reload best checkpoint. Write `pred_A2B` (photo→Monet) and `pred_B2A` (Monet→photo). For Kaggle, zip photo→Monet JPGs as `images.zip` (7k–10k for full).


In [8]:
# reload best and run inference
state = torch.load(ckpt_path, map_location=device, weights_only=True)
G_AB.load_state_dict(state["G_AB"])
G_BA.load_state_dict(state["G_BA"])
G_AB.eval(); G_BA.eval()


def tensor_to_pil(t):
    t = t.detach().cpu().clamp(-1, 1)
    t = (t + 1) * 0.5
    arr = (t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(arr)


@torch.no_grad()
def translate_folder(gen, paths, out_dir, limit):
    out_dir = Path(out_dir)
    for p in out_dir.glob("*"):
        if p.is_file() and p.name != ".gitkeep":
            p.unlink()
    ds = ImageFolderDS(paths[:limit], train=False)
    saved = 0
    for i in range(len(ds)):
        x = ds[i].unsqueeze(0).to(device)
        y = gen(x)[0]
        tensor_to_pil(y).save(out_dir / ("%05d.jpg" % i), quality=95)
        saved += 1
    return saved


# A2B = photo -> monet (Kaggle direction), B2A = monet -> photo
n_a2b = translate_folder(G_AB, photo_paths, PRED_A2B, N_SUBMIT if not SMOKE else min(N_SUBMIT, len(photo_paths)))
n_b2a = translate_folder(G_BA, monet_paths, PRED_B2A, min(len(monet_paths), 32 if SMOKE else len(monet_paths)))
print("wrote pred_A2B:", n_a2b, "pred_B2A:", n_b2a)

# sample grid
n_show = min(int(CFG["n_sample_grid"]), len(photo_paths), len(monet_paths))
fig, axes = plt.subplots(n_show, 4, figsize=(10, 2.4 * n_show))
if n_show == 1:
    axes = np.expand_dims(axes, 0)
photo_eval = ImageFolderDS(photo_paths, train=False)
monet_eval = ImageFolderDS(monet_paths, train=False)
with torch.no_grad():
    for r in range(n_show):
        pb = photo_eval[r].unsqueeze(0).to(device)
        ma = monet_eval[r].unsqueeze(0).to(device)
        fake_m = G_AB(pb)[0]
        fake_p = G_BA(ma)[0]
        for c, tens in enumerate([pb[0], fake_m, ma[0], fake_p]):
            axes[r, c].imshow(tensor_to_pil(tens))
            axes[r, c].axis("off")
axes[0, 0].set_title("photo")
axes[0, 1].set_title("→ Monet")
axes[0, 2].set_title("Monet")
axes[0, 3].set_title("→ photo")
fig.tight_layout()
grid_path = OUT / "samples" / ("grid_smoke.png" if SMOKE else "grid.png")
fig.savefig(grid_path, dpi=120)
plt.close(fig)
print("saved", relpath(grid_path))

# Kaggle images.zip from pred_A2B
import zipfile
zip_path = MEMBER / ("images_smoke.zip" if SMOKE else "images.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for p in sorted(PRED_A2B.glob("*.jpg")):
        zf.write(p, arcname=p.name)
print("kaggle zip:", relpath(zip_path), "files:", n_a2b)


wrote pred_A2B: 7038 pred_B2A: 300


saved task3_gan/sneha_singh/outputs/samples/grid.png


kaggle zip: task3_gan/sneha_singh/images.zip files: 7038


In [9]:
# lightweight cycle-L1 check + write metrics stub rows + manifest
def cycle_l1(n=min(16, len(photo_paths), len(monet_paths))):
    photo_eval = ImageFolderDS(photo_paths, train=False)
    monet_eval = ImageFolderDS(monet_paths, train=False)
    s_ab = s_ba = 0.0
    with torch.no_grad():
        for i in range(n):
            b = photo_eval[i].unsqueeze(0).to(device)
            a = monet_eval[i].unsqueeze(0).to(device)
            rec_b = G_BA(G_AB(b))
            rec_a = G_AB(G_BA(a))
            s_ab += F.l1_loss(rec_b, b).item()
            s_ba += F.l1_loss(rec_a, a).item()
    return s_ab / n, s_ba / n


c_photo, c_monet = cycle_l1()
print("cycle L1 photo→M→photo:", round(c_photo, 4))
print("cycle L1 Monet→P→Monet:", round(c_monet, 4))

ips = total_images / max(1e-9, train_time)  # actual images seen in train loop
fields = [
    "direction","fid","kid","precision","recall","cycle_l1","lpips","content_cosine",
    "g_loss","d_loss","cycle_loss","identity_loss","grad_norm","nan_count","param_count",
    "train_time_sec","images_per_sec","peak_memory_mb","human_audit_score",
    "inter_rater_agreement","kaggle_public","kaggle_private","kaggle_rank",
]
rows = [
    {
        "direction": "A2B", "cycle_l1": c_photo,
        "g_loss": history["g"][-1], "d_loss": history["d"][-1],
        "cycle_loss": history["cycle"][-1], "identity_loss": history["identity"][-1],
        "grad_norm": history["grad"][-1], "nan_count": nan_count,
        "param_count": param_count, "train_time_sec": train_time,
        "images_per_sec": ips, "peak_memory_mb": peak_mem,
    },
    {
        "direction": "B2A", "cycle_l1": c_monet,
        "g_loss": history["g"][-1], "d_loss": history["d"][-1],
        "cycle_loss": history["cycle"][-1], "identity_loss": history["identity"][-1],
        "grad_norm": history["grad"][-1], "nan_count": nan_count,
        "param_count": param_count, "train_time_sec": train_time,
        "images_per_sec": ips, "peak_memory_mb": peak_mem,
    },
]
for r in rows:
    for k in fields:
        r.setdefault(k, "")

metrics_path = MEMBER / "full_metrics_report.csv"
with open(metrics_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
print("wrote", relpath(metrics_path))
print("run evaluate_local.py later for FID/KID/LPIPS after deps are installed")


cycle L1 photo→M→photo: 0.1167
cycle L1 Monet→P→Monet: 0.1025
wrote task3_gan/sneha_singh/full_metrics_report.csv
run evaluate_local.py later for FID/KID/LPIPS after deps are installed


In [10]:
# append reproducibility manifest
import platform
import sys
from datetime import datetime

hw = str(device)
if device.type == "cuda":
    hw = "cuda: " + torch.cuda.get_device_name(0)
elif device.type == "mps":
    hw = "mps (Apple Silicon)"

manifest_path = REPO / "reproducibility" / "manifests" / "sneha_singh.md"
section = "\n".join([
    "",
    "## Run — Part 3 CycleGAN (%s)" % ("smoke" if SMOKE else "full"),
    "- Date: %s" % datetime.now().strftime("%Y-%m-%d %H:%M"),
    "- Task: task3_gan",
    "- Smoke: %s" % SMOKE,
    "- Checkpoint: %s" % relpath(ckpt_path),
    "- Pred A2B: %s" % relpath(PRED_A2B),
    "- Pred B2A: %s" % relpath(PRED_B2A),
    "- Metrics CSV: %s" % relpath(MEMBER / "full_metrics_report.csv"),
    "- Raw log: %s" % relpath(log_path),
    "- Python: %s" % sys.version.split()[0],
    "- PyTorch: %s" % torch.__version__,
    "- Device: %s" % hw,
    "- Platform: %s" % platform.platform(),
    "- Command: Run All on task3_gan/sneha_singh/src/part3_cyclegan.ipynb (config.json smoke=%s)" % str(SMOKE).lower(),
    "",
])
header = "# Environment manifest — Sneha Singh\n\nDo not rewrite history; add a new section per run.\n"
if manifest_path.exists() and manifest_path.read_text().strip().startswith("# Environment manifest"):
    manifest_path.write_text(manifest_path.read_text().rstrip() + "\n" + section)
else:
    manifest_path.write_text(header + section)
print("wrote/appended", relpath(manifest_path))
print("Part 3 smoke/full training cell done")


wrote/appended reproducibility/manifests/sneha_singh.md
Part 3 smoke/full training cell done
